In [ ]:
import subprocess
import textwrap
import base64



# ----------------------------
# Low-level SSH runner
# ----------------------------
def g5k(cmd: str) -> str:
    p = subprocess.run(
        ["ssh", "YOUR_SSH_HOST", "bash", "-s"],
        input=cmd,
        text=True,
        capture_output=True,
    )
    if p.returncode != 0:
        raise RuntimeError(
            f"Remote cmd failed (rc={p.returncode}).\n"
            f"---CMD---\n{cmd}\n"
            f"---STDERR---\n{p.stderr}"
        )
    return p.stdout

# ----------------------------
# Remote environment wrapper
# ----------------------------
def g5k_tmp(cmd: str) -> str:
    preamble = r'''
set -e

export TMPBASE="${TMPDIR:-/tmp}/$USER"
export HF_HOME="$TMPBASE/hf-cache"
export TRANSFORMERS_CACHE="$HF_HOME"
export HF_DATASETS_CACHE="$TMPBASE/hf-datasets"
export HUGGINGFACE_HUB_CACHE="$HF_HOME/hub"

mkdir -p "$TMPBASE" "$HF_HOME" "$HF_DATASETS_CACHE" "$HUGGINGFACE_HUB_CACHE"

# remove old quota-filling cache if it reappears
rm -rf "$HOME/.cache/huggingface_impresso" 2>/dev/null || true

umask 077
'''
    return g5k(preamble + "\n" + cmd)

# ----------------------------
# One-time setup
# ----------------------------
setup_out = g5k_tmp(r'''
echo "HOST=$(hostname)"
echo "HOME=$HOME"
echo "TMPBASE=$TMPBASE"
echo "HF_HOME=$HF_HOME"
echo "HF_DATASETS_CACHE=$HF_DATASETS_CACHE"
echo
echo "[home usage]"
du -sh "$HOME" 2>/dev/null || true
echo
echo "[scratch usage before]"
du -sh "$TMPBASE" 2>/dev/null || true
WORKDIR=$(mktemp -d -p "$TMPBASE" qa_run.XXXXXX)
echo
echo "WORKDIR=$WORKDIR"
''')

print(setup_out)

workdir = None
for line in setup_out.splitlines():
    if line.startswith("WORKDIR="):
        workdir = line.split("=", 1)[1].strip()
        break

if not workdir:
    raise RuntimeError("Failed to create remote WORKDIR")

print("Python variable ready:")
print("workdir =", workdir)

# ----------------------------
# Helper: run bash inside remote workdir
# ----------------------------
def run_remote(script: str) -> str:
    remote_cmd = f'''
cd "{workdir}"
{script}
'''
    return g5k_tmp(remote_cmd)

# ----------------------------
# Helper: write a file into remote workdir
# ----------------------------
def write_remote_file(filename: str, content: str) -> str:
    payload = base64.b64encode(content.encode("utf-8")).decode("ascii")
    cmd = f'''
cd "{workdir}"
python - <<'PY'
import base64
content = base64.b64decode("{payload}").decode("utf-8")
with open("{filename}", "w", encoding="utf-8") as f:
    f.write(content)
print("WROTE", "{filename}")
PY
'''
    return g5k_tmp(cmd)

# ----------------------------
# Smoke test
# ----------------------------
print(run_remote(r'''
pwd
echo
ls -la
echo
which python || true
python --version || true
nvidia-smi || true
'''))

print(run_remote(r'''
echo "[free space]"
df -h .
df -h "$TMPBASE"
df -h "$HOME"
'''))

print(run_remote(r'''
echo "[free inodes]"
df -ih .
df -ih "$TMPBASE"
df -ih "$HOME"
'''))


# install (use terminal)

In [ ]:
# sudo -u projectuser -H bash -lc '
# mkdir -p /home/USER/venvs
# python3 -m venv /home/USER/venvs/audit-env
# source /home/USER/venvs/audit-env/bin/activate
# python -m pip install --upgrade pip wheel setuptools
# python -m pip install --upgrade ipykernel sentence-transformers qdrant-client
# python -m ipykernel install --user --name projectuser-audit-env --display-name "Python (projectuser-audit)"
# '

In [ ]:
# sudo systemctl restart jupyterhub
# sudo systemctl status jupyterhub --no-pager

In [ ]:
# import sys
# print(sys.executable)

# from sentence_transformers import SentenceTransformer
# import qdrant_client

# print("Imports OK")


# prompts 

In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import gc
import json
import re
import shutil
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple


# =========================================================
# JDH ROOT / DEFAULT PATHS
# =========================================================
def resolve_jdh_root() -> Path:
    candidates = [
        Path("./")
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return Path("/")


JDH_ROOT = resolve_jdh_root()

# PRIMARY_REVIEWS_ROOT = JDH_ROOT / "review_outputs" / "paper_reviews"
# FALLBACK_REVIEWS_ROOT = JDH_ROOT / "paper_reviews"

PRIMARY_REVIEWS_ROOT = JDH_ROOT / "review_outputs" / "separated_reviews" / "review_outputs" / "paper_reviews"
FALLBACK_REVIEWS_ROOT = JDH_ROOT / "review_outputs" / "paper_reviews"
SECOND_FALLBACK_REVIEWS_ROOT = JDH_ROOT / "paper_reviews"


PRIMARY_LOCAL_QDRANT_DIR = JDH_ROOT / "review_outputs" / "paper_qdrant_db"
FALLBACK_LOCAL_QDRANT_DIR = JDH_ROOT / "paper_qdrant_db"

PRIMARY_RESULTS_ROOT = JDH_ROOT / "g5k_paper_chunk_results"
FALLBACK_RESULTS_ROOT = Path("g5k_paper_chunk_results")

QDRANT_RUNTIME_ROOT = JDH_ROOT / "_qdrant_runtime"


# =========================================================
# SETTINGS
# =========================================================
EMBEDDING_MODEL_ID = "BAAI/bge-large-en-v1.5"

REVIEWS_ROOT = PRIMARY_REVIEWS_ROOT
LOCAL_QDRANT_DIR = PRIMARY_LOCAL_QDRANT_DIR
DEFAULT_OUTPUT_DIR = JDH_ROOT / "prepared_audit_prompts"

QDRANT_MODE = "local"   # local | server
QDRANT_URL = ""
QDRANT_API_KEY = ""

TOP_K = 5

# Minimum comment length to be considered non-trivial for retrieval
MIN_COMMENT_LENGTH = 20

# Patterns that indicate a trivial / empty comment
TRIVIAL_COMMENT_PATTERNS = [
    r"(?i)^n/?a$",
    r"(?i)^none\.?$",
    r"(?i)^no\s+comment",
    r"(?i)^i\s+have\s+no\s+(additional\s+)?comment",
    r"(?i)^nothing\s+(to\s+add|else|further)",
    r"(?i)^no\s+(additional|further)\s+(comment|remark|feedback)",
    r"(?i)^-+$",
    r"(?i)^\.+$",
    r"(?i)^see\s+above",
    r"(?i)^no\.?$",
    r"(?i)^not relevant\.?$",
]


# =========================================================
# PROMPTS / SCHEMAS
# =========================================================
COMMENT_SYSTEM_PROMPT = """You are an evidence-grounded academic review auditor.

You receive one review question, one reviewer comment for that question, and retrieved paper chunks.
Judge the reviewer comment only against the supplied chunks.
Be conservative.
Do not use outside knowledge.
Prefer exact evidence from the provided chunks.
If evidence is weak or missing, say so.
Return exactly one JSON object and nothing else.
"""

COMMENT_SCHEMA_HINT: Dict[str, Any] = {
    "comment_id": "string",
    "question": "string",
    "reviewer_comment": "string",
    "support_label": "supported | partially_supported | not_supported | insufficient_evidence",
    "confidence": 0.0,
    "evidence": [
        {
            "chunk_id": "string",
            "section": "string",
            "page_start": 0,
            "page_end": 0,
            "snippet": "string",
        }
    ],
    "reasoning_summary": "string",
    "retrieval_strength": 0.0,
    "evidence_specificity": 0.0,
    "section_coverage": 0.0,
}


# =========================================================
# DATA CLASSES
# =========================================================
@dataclass
class PaperEntry:
    paper_id: str
    collection_name: str
    source_file: str = ""


# =========================================================
# TEXT HELPERS
# =========================================================
def normalize_whitespace(text: Any) -> str:
    if text is None:
        return ""
    text = str(text).replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_text(text: Any) -> str:
    return normalize_whitespace(text)


def first_nonempty(*values: Any) -> str:
    for value in values:
        txt = normalize_whitespace(value)
        if txt:
            return txt
    return ""


def shorten(text: Any, limit: int = 140) -> str:
    txt = clean_text(text)
    if len(txt) <= limit:
        return txt
    return txt[: limit - 3] + "..."


def truncate_for_judge(text: Any, max_chars: int = 900) -> str:
    txt = clean_text(text)
    if len(txt) <= max_chars:
        return txt
    return txt[:max_chars] + " ..."


def canonical_key(value: Any) -> str:
    value = normalize_whitespace(value).lower()
    value = Path(value).stem
    value = re.sub(r"(?:_bundle_clean|_clean|_bundle|_paper|_chunks?)$", "", value)
    value = re.sub(r"[^a-z0-9]+", "", value)
    return value


def extract_numeric_id(value: Any) -> str:
    m = re.search(r"(\d+)", str(value))
    return str(int(m.group(1))) if m else ""


def looks_like_score(text: str) -> bool:
    s = clean_text(text).lower()
    if not s:
        return False
    if re.match(r"^[0-9]+(?:\.[0-9]+)?(?:\s*/\s*[0-9]+(?:\.[0-9]+)?)?$", s):
        return True
    return s in {
        "strong reject",
        "reject",
        "weak reject",
        "borderline",
        "weak accept",
        "accept",
        "strong accept",
    }


def is_trivial_comment(text: str) -> bool:
    txt = clean_text(text)
    if len(txt) < MIN_COMMENT_LENGTH:
        return True
    for pat in TRIVIAL_COMMENT_PATTERNS:
        if re.match(pat, txt):
            return True
    return False


# =========================================================
# PATH RESOLUTION
# =========================================================
def resolve_reviews_root(reviews_root: Path) -> Path:
    candidates: List[Path] = []

    if reviews_root.is_absolute():
        candidates.append(reviews_root)
    else:
        candidates.extend([
            reviews_root,
            Path.cwd() / reviews_root,
            JDH_ROOT / reviews_root,
            JDH_ROOT / "review_outputs" / reviews_root.name,
            JDH_ROOT / "review_outputs" / "paper_reviews",
            JDH_ROOT / "paper_reviews",
        ])

    seen = set()
    ordered: List[Path] = []
    for candidate in candidates:
        resolved = candidate.resolve()
        if str(resolved) not in seen:
            seen.add(str(resolved))
            ordered.append(resolved)

    for candidate in ordered:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find reviews root. Tried:\n" +
        "\n".join(f" - {p}" for p in ordered)
    )


def resolve_local_qdrant_dir(local_qdrant_dir: Path) -> Path:
    candidates: List[Path] = []

    if local_qdrant_dir.is_absolute():
        candidates.append(local_qdrant_dir)
    else:
        candidates.extend([
            local_qdrant_dir,
            Path.cwd() / local_qdrant_dir,
            JDH_ROOT / local_qdrant_dir,
            JDH_ROOT / "review_outputs" / local_qdrant_dir.name,
            JDH_ROOT / "review_outputs" / "paper_qdrant_db",
            JDH_ROOT / "paper_qdrant_db",
        ])

    seen = set()
    ordered: List[Path] = []
    for candidate in candidates:
        resolved = candidate.resolve()
        if str(resolved) not in seen:
            seen.add(str(resolved))
            ordered.append(resolved)

    for candidate in ordered:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find local Qdrant dir. Tried:\n" +
        "\n".join(f" - {p}" for p in ordered)
    )


def resolve_latest_summary_json() -> Path:
    search_roots = [
        PRIMARY_RESULTS_ROOT,
        FALLBACK_RESULTS_ROOT,
    ]

    found: List[Path] = []
    for base in search_roots:
        if not base.exists():
            continue
        job_dirs = [p for p in base.glob("job_*") if p.is_dir()]
        for job_dir in job_dirs:
            candidate = job_dir / "summary.json"
            if candidate.exists():
                found.append(candidate.resolve())

    if not found:
        raise FileNotFoundError(
            "Could not find latest summary.json under:\n"
            f" - {PRIMARY_RESULTS_ROOT}\n"
            f" - {FALLBACK_RESULTS_ROOT}"
        )

    return max(found, key=lambda p: p.stat().st_mtime)


# =========================================================
# LOCAL DISCOVERY / MAPPING
# =========================================================
def load_summary_registry(summary_path: Path) -> Dict[str, PaperEntry]:
    obj = json.loads(summary_path.read_text(encoding="utf-8"))
    rows = obj.get("results", []) if isinstance(obj, dict) else obj

    registry: Dict[str, PaperEntry] = {}
    for idx, row in enumerate(rows, start=1):
        if not isinstance(row, dict):
            continue

        paper_id = first_nonempty(row.get("paper_id"), str(idx))
        paper_id = extract_numeric_id(paper_id) or paper_id
        collection_name = first_nonempty(row.get("collection_name"), f"paper_{paper_id}")

        registry[paper_id] = PaperEntry(
            paper_id=paper_id,
            collection_name=collection_name,
            source_file=first_nonempty(row.get("source_file")),
        )

    if not registry:
        raise RuntimeError(f"No paper entries found in {summary_path}")

    return registry


def discover_review_json_files(root: Path) -> List[Path]:
    root = resolve_reviews_root(root)

    files = sorted({
        *root.rglob("*.json"),
    })

    files = [
        p for p in files
        if ".ipynb_checkpoints" not in p.parts
        and p.is_file()
    ]

    if not files:
        raise RuntimeError(f"No review JSON files found under {root.resolve()}")

    print("[DEBUG] found review JSON files:")
    for p in files:
        print(" -", p)

    return files
    


def resolve_review_paper(review_file: Path, registry: Dict[str, PaperEntry]) -> PaperEntry:
    candidates: List[str] = []

    for item in [
        review_file.name,
        review_file.stem,
        review_file.parent.name,
        review_file.parent.parent.name,
    ]:
        pid = extract_numeric_id(item)
        if pid:
            candidates.append(pid)

    for pid in candidates:
        if pid in registry:
            return registry[pid]

    review_keys = {
        canonical_key(review_file.name),
        canonical_key(review_file.stem),
        canonical_key(review_file.parent.name),
        canonical_key(review_file.parent.parent.name),
    }

    matches = []
    for entry in registry.values():
        entry_keys = {
            canonical_key(entry.paper_id),
            canonical_key(entry.collection_name),
            canonical_key(entry.source_file),
            canonical_key(Path(entry.source_file).name if entry.source_file else ""),
        }
        if review_keys & entry_keys:
            matches.append(entry)

    if len(matches) == 1:
        return matches[0]

    raise RuntimeError(f"Could not uniquely map review file to paper: {review_file}")


# =========================================================
# REVIEW PARSING
# =========================================================
def extract_score_raw(review_obj: Any) -> str:
    if isinstance(review_obj, dict):
        for key in ["score", "overall_score", "overall_rating", "recommendation", "final_score", "score_raw"]:
            value = first_nonempty(review_obj.get(key))
            if value:
                return value

        items = review_obj.get("items", [])
        if isinstance(items, list):
            for item in items:
                if not isinstance(item, dict):
                    continue

                question = first_nonempty(item.get("question"), item.get("prompt"), item.get("title"))
                answer = first_nonempty(
                    item.get("reviewer_comment"),
                    item.get("comment"),
                    item.get("answer"),
                    item.get("response"),
                    item.get("text"),
                    item.get("value"),
                )

                if question and answer and re.search(r"\b(score|rating|recommendation|overall)\b", question, flags=re.IGNORECASE):
                    if looks_like_score(answer):
                        return answer
    return ""


def compose_retrieval_query(question: str, reviewer_comment: str) -> str:
    comment = clean_text(reviewer_comment)

    # retrieve using the reviewer comment whenever it is non-trivial
    if comment and not is_trivial_comment(comment):
        return comment

    # only fallback to the question when the comment is trivial/empty
    q = clean_text(question)
    if q:
        return q

    return comment


def parse_review_file(review_path: Path) -> Dict[str, Any]:
    obj = json.loads(review_path.read_text(encoding="utf-8"))

    reviewer_id = (
        first_nonempty(obj.get("reviewer_id"), obj.get("reviewer"), review_path.stem)
        if isinstance(obj, dict)
        else review_path.stem
    )

    score_raw = extract_score_raw(obj)
    items = obj.get("items", []) if isinstance(obj, dict) else obj if isinstance(obj, list) else []

    comments = []

    for idx, item in enumerate(items, start=1):
        if not isinstance(item, dict):
            continue

        question = first_nonempty(
            item.get("question"),
            item.get("prompt"),
            item.get("criterion"),
            item.get("title"),
        )

        reviewer_comment = first_nonempty(
            item.get("reviewer_comment"),
            item.get("comment"),
            item.get("answer"),
            item.get("response"),
            item.get("text"),
            item.get("value"),
        )

        # Skip score/rating-style items from comment auditing
        if question and reviewer_comment and re.search(
            r"\b(score|rating|recommendation|overall)\b",
            question,
            flags=re.IGNORECASE,
        ):
            if looks_like_score(reviewer_comment):
                if not score_raw:
                    score_raw = reviewer_comment
                continue

        reviewer_comment = clean_text(reviewer_comment)
        if not reviewer_comment:
            continue

        comments.append(
            {
                "comment_id": str(first_nonempty(item.get("item_id"), item.get("id"), f"C{idx}")),
                "question": question,
                "reviewer_comment": reviewer_comment,
                "retrieval_query": compose_retrieval_query(question, reviewer_comment),
            }
        )

    return {
        "reviewer_id": reviewer_id,
        "score_raw": score_raw,
        "comments": comments,
        "source_path": str(review_path),
    }


# =========================================================
# LOCAL EMBEDDING / QDRANT
# =========================================================
class LocalEmbedder:
    def __init__(self, model_id: str, batch_size: int = 16) -> None:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as exc:
            raise RuntimeError(
                "Missing local dependency: sentence-transformers. "
                "Install it before running this script."
            ) from exc

        print(f"[LOCAL EMBEDDER] loading {model_id}", flush=True)
        self.model = SentenceTransformer(model_id, device="cpu")
        self.batch_size = batch_size
        print("[LOCAL EMBEDDER] ready", flush=True)

    def encode(self, texts: Sequence[str]) -> Any:
        return self.model.encode(
            list(texts),
            batch_size=self.batch_size,
            show_progress_bar=False,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )


def _copy_qdrant_snapshot(src_dir: Path) -> Path:
    QDRANT_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)

    snapshot_root = Path(tempfile.mkdtemp(prefix="qdrant_snapshot_", dir=str(QDRANT_RUNTIME_ROOT)))
    snapshot_dir = snapshot_root / src_dir.name

    shutil.copytree(
        src_dir,
        snapshot_dir,
        dirs_exist_ok=True,
        ignore=shutil.ignore_patterns(".lock", "*.lock", "lock", "*.tmp"),
    )

    for pattern in [".lock", "*.lock", "lock", "*.tmp"]:
        for p in snapshot_dir.rglob(pattern):
            if p.is_file():
                try:
                    p.unlink()
                except Exception:
                    pass

    return snapshot_root


def make_local_qdrant_client(
    qdrant_mode: str,
    local_qdrant_dir: Path,
    qdrant_url: str,
    qdrant_api_key: str,
) -> Tuple[Any, Path | None]:
    try:
        from qdrant_client import QdrantClient
    except ImportError as exc:
        raise RuntimeError(
            "Missing local dependency: qdrant-client. "
            "Install it before running this script."
        ) from exc

    if qdrant_mode == "server":
        if not qdrant_url:
            raise ValueError("QDRANT_MODE is 'server' but QDRANT_URL is empty.")
        kwargs: Dict[str, Any] = {"url": qdrant_url}
        if qdrant_api_key:
            kwargs["api_key"] = qdrant_api_key
        return QdrantClient(**kwargs), None

    src_dir = resolve_local_qdrant_dir(local_qdrant_dir)
    snapshot_root = _copy_qdrant_snapshot(src_dir)
    snapshot_dir = snapshot_root / src_dir.name

    print(f"[LOCAL PREP] using qdrant snapshot: {snapshot_dir}", flush=True)
    client = QdrantClient(path=str(snapshot_dir))
    return client, snapshot_root


def collection_exists_local(client: Any, collection_name: str) -> bool:
    return collection_name in [c.name for c in client.get_collections().collections]


def extract_payload_fields(payload: Dict[str, Any]) -> Dict[str, Any]:
    metadata = payload.get("metadata", {}) if isinstance(payload.get("metadata"), dict) else {}

    def pick(*keys: str, default: Any = "") -> Any:
        for key in keys:
            if key in payload and payload[key] not in (None, ""):
                return payload[key]
            if key in metadata and metadata[key] not in (None, ""):
                return metadata[key]
        return default

    return {
        "chunk_id": clean_text(pick("chunk_id", default="")),
        "section": clean_text(
            pick("section_title", "section", default="unknown")
        ),
        "page_start": int(pick("page_start", default=0) or 0),
        "page_end": int(pick("page_end", default=0) or 0),
        "text": clean_text(pick("text", default="")),
    }


def search_top_k_chunks_local(
    client: Any,
    embedder: LocalEmbedder,
    collection_name: str,
    query_text: str,
    top_k: int,
) -> List[Dict[str, Any]]:
    if not collection_exists_local(client, collection_name):
        raise ValueError(f"Missing Qdrant collection: {collection_name}")

    vector = embedder.encode([query_text])[0].tolist()

    try:
        hits = client.query_points(
            collection_name=collection_name,
            query=vector,
            limit=top_k,
            with_payload=True,
        )
        points = getattr(hits, "points", hits)
    except Exception:
        points = client.search(
            collection_name=collection_name,
            query_vector=vector,
            limit=top_k,
            with_payload=True,
        )

    out: List[Dict[str, Any]] = []
    for hit in points:
        payload = hit.payload if hasattr(hit, "payload") else hit.get("payload", {})
        score = hit.score if hasattr(hit, "score") else hit.get("score", 0.0)
        row = extract_payload_fields(payload)
        row["retrieval_score"] = float(score)
        out.append(row)

    return out


# =========================================================
# PROMPT BUILDING
# =========================================================
def build_comment_messages(
    paper_id: str,
    paper_title: str,
    reviewer_id: str,
    comment: Dict[str, Any],
    retrieved_chunks: List[Dict[str, Any]],
) -> List[Dict[str, str]]:
    payload = {
        "paper_id": paper_id,
        "paper_title": paper_title,
        "reviewer_id": reviewer_id,
        "comment_id": comment["comment_id"],
        "question": comment.get("question", ""),
        "reviewer_comment": comment["reviewer_comment"],
        "retrieved_chunks": [
            {
                "chunk_id": c["chunk_id"],
                "section": c.get("section", "unknown"),
                "page_start": c.get("page_start", 0),
                "page_end": c.get("page_end", 0),
                "retrieval_score": c.get("retrieval_score", 0.0),
                "text": truncate_for_judge(c.get("text", ""), 900),
            }
            for c in retrieved_chunks
        ],
    }

    user_prompt = (
        "Return exactly one JSON object.\n"
        "Do not wrap in markdown.\n"
        "Do not add commentary.\n"
        "Use double quotes for every key and string value.\n"
        "Start with { and end with }.\n\n"
        "Use these label meanings exactly:\n"
        "- supported: the claim is directly backed by the retrieved chunks.\n"
        "- partially_supported: the claim is somewhat backed but important parts are missing, overstated, or only indirectly supported.\n"
        "- not_supported: the claim conflicts with, or is clearly not backed by, the retrieved chunks.\n"
        "- insufficient_evidence: the retrieved chunks are too weak or too incomplete to decide confidently.\n\n"
        "Required JSON shape:\n"
        f"{json.dumps(COMMENT_SCHEMA_HINT, ensure_ascii=False, indent=2)}\n\n"
        "Input:\n"
        f"{json.dumps(payload, ensure_ascii=False, indent=2)}"
    )

    return [
        {"role": "system", "content": COMMENT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


# =========================================================
# PREPARE LOCAL PROMPT BUNDLE
# =========================================================
def prepare_reviews_payload(
    summary_json: Path,
    reviews_root: Path,
    local_qdrant_dir: Path,
    qdrant_mode: str,
    qdrant_url: str,
    qdrant_api_key: str,
    top_k: int,
) -> List[Dict[str, Any]]:
    registry = load_summary_registry(summary_json)
    review_files = discover_review_json_files(reviews_root)

    embedder: LocalEmbedder | None = None
    qdrant: Any = None
    qdrant_snapshot_DEMO: Path | None = None

    try:
        print(f"[LOCAL PREP] loading embedder: {EMBEDDING_MODEL_ID}", flush=True)
        embedder = LocalEmbedder(EMBEDDING_MODEL_ID)

        print(f"[LOCAL PREP] opening qdrant mode={qdrant_mode}", flush=True)
        qdrant, qdrant_snapshot_DEMO = make_local_qdrant_client(
            qdrant_mode=qdrant_mode,
            local_qdrant_dir=local_qdrant_dir,
            qdrant_url=qdrant_url,
            qdrant_api_key=qdrant_api_key,
        )

        prepared_reviews: List[Dict[str, Any]] = []
        total_reviews = len(review_files)
        print(f"[LOCAL PREP] review files = {total_reviews}", flush=True)

        for review_idx, review_file in enumerate(review_files, start=1):
            paper = resolve_review_paper(review_file, registry)
            parsed = parse_review_file(review_file)

            reviewer_id = parsed["reviewer_id"]
            score_raw = parsed["score_raw"]
            comments = parsed["comments"]

            if not comments:
                raise RuntimeError(f"No usable comments found in {review_file}")

            paper_id = paper.paper_id
            collection_name = paper.collection_name
            paper_title = Path(paper.source_file or f"paper_{paper_id}").stem

            print(
                f"[LOCAL PREP {review_idx}/{total_reviews}] "
                f"paper_id={paper_id} reviewer={reviewer_id} comments={len(comments)} "
                f"collection={collection_name}",
                flush=True,
            )

            comment_tasks: List[Dict[str, Any]] = []

            for comment_idx, comment in enumerate(comments, start=1):
                retrieval_query = comment["retrieval_query"]
                is_fallback = is_trivial_comment(comment["reviewer_comment"])
                query_source = "FALLBACK:question" if is_fallback else "comment"

                print(
                    f"  [RETRIEVE {comment_idx}/{len(comments)}] "
                    f"id={comment['comment_id']} query_source={query_source} "
                    f"query={shorten(retrieval_query, 100)}",
                    flush=True,
                )

                hits = search_top_k_chunks_local(
                    client=qdrant,
                    embedder=embedder,
                    collection_name=collection_name,
                    query_text=retrieval_query,
                    top_k=top_k,
                )

                messages = build_comment_messages(
                    paper_id=paper_id,
                    paper_title=paper_title,
                    reviewer_id=reviewer_id,
                    comment=comment,
                    retrieved_chunks=hits,
                )

                comment_tasks.append(
                    {
                        "comment_id": comment["comment_id"],
                        "question": comment.get("question", ""),
                        "reviewer_comment": comment["reviewer_comment"],
                        "retrieval_query": retrieval_query,
                        "retrieval_query_source": query_source,
                        "retrieved_chunks": [
                            {
                                "chunk_id": c["chunk_id"],
                                "section": c.get("section", "unknown"),
                                "page_start": int(c.get("page_start", 0) or 0),
                                "page_end": int(c.get("page_end", 0) or 0),
                                "retrieval_score": float(c.get("retrieval_score", 0.0) or 0.0),
                                "full_chunk_text": clean_text(c.get("text", "")),
                            }
                            for c in hits
                        ],
                        "messages": messages,
                    }
                )

            prepared_reviews.append(
                {
                    "paper_id": paper_id,
                    "paper_title": paper_title,
                    "collection_name": collection_name,
                    "source_file": paper.source_file,
                    "reviewer_id": reviewer_id,
                    "review_source_path": str(review_file),
                    "score_raw": score_raw,
                    "top_k": top_k,
                    "comment_tasks": comment_tasks,
                }
            )

        return prepared_reviews

    finally:
        if qdrant is not None:
            try:
                qdrant.close()
            except Exception:
                pass

        try:
            del qdrant
        except Exception:
            pass

        try:
            del embedder
        except Exception:
            pass

        gc.collect()

        if qdrant_snapshot_DEMO is not None and qdrant_snapshot_DEMO.exists():
            shutil.rmtree(qdrant_snapshot_DEMO, ignore_errors=True)


# =========================================================
# SAVE OUTPUTS
# =========================================================
def save_outputs(prepared_reviews: List[Dict[str, Any]], output_dir: Path) -> Dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    prepared_reviews_path = output_dir / "prepared_reviews.json"
    comment_prompts_jsonl_path = output_dir / "comment_prompts.jsonl"

    prepared_reviews_path.write_text(
        json.dumps(prepared_reviews, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    flat_count = 0
    with comment_prompts_jsonl_path.open("w", encoding="utf-8") as f:
        for review in prepared_reviews:
            for task in review["comment_tasks"]:
                row = {
                    "paper_id": review["paper_id"],
                    "paper_title": review["paper_title"],
                    "collection_name": review["collection_name"],
                    "source_file": review.get("source_file", ""),
                    "reviewer_id": review["reviewer_id"],
                    "review_source_path": review["review_source_path"],
                    "score_raw": review.get("score_raw", ""),
                    "top_k": review["top_k"],
                    "comment_id": task["comment_id"],
                    "question": task.get("question", ""),
                    "reviewer_comment": task["reviewer_comment"],
                    "retrieval_query": task["retrieval_query"],
                    "retrieval_query_source": task["retrieval_query_source"],
                    "retrieved_chunks": task["retrieved_chunks"],
                    "messages": task["messages"],
                }
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                flat_count += 1

    manifest = {
        "prepared_reviews_json": str(prepared_reviews_path),
        "comment_prompts_jsonl": str(comment_prompts_jsonl_path),
        "review_count": len(prepared_reviews),
        "comment_prompt_count": flat_count,
    }

    manifest_path = output_dir / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    return {
        "prepared_reviews_json": prepared_reviews_path,
        "comment_prompts_jsonl": comment_prompts_jsonl_path,
        "manifest_json": manifest_path,
    }


# =========================================================
# CLI
# =========================================================
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Prepare all comment-level judge prompts locally using local Qdrant retrieval."
    )
    parser.add_argument(
        "--summary-json",
        type=Path,
        default=None,
        help="Path to summary.json. Default: latest JDH/g5k_paper_chunk_results/job_*/summary.json",
    )
    parser.add_argument(
        "--reviews-root",
        type=Path,
        default=REVIEWS_ROOT,
        help="Root containing separated review JSON files",
    )
    parser.add_argument(
        "--local-qdrant-dir",
        type=Path,
        default=LOCAL_QDRANT_DIR,
        help="Local Qdrant directory",
    )
    parser.add_argument(
        "--qdrant-mode",
        type=str,
        default=QDRANT_MODE,
        choices=["local", "server"],
        help="Qdrant mode",
    )
    parser.add_argument(
        "--qdrant-url",
        type=str,
        default=QDRANT_URL,
        help="Qdrant server URL",
    )
    parser.add_argument(
        "--qdrant-api-key",
        type=str,
        default=QDRANT_API_KEY,
        help="Qdrant server API key",
    )
    parser.add_argument(
        "--top-k",
        type=int,
        default=TOP_K,
        help="Top-K chunks per comment",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=DEFAULT_OUTPUT_DIR,
        help="Directory to write prepared prompt files",
    )

    import sys
    if "ipykernel" in sys.modules:
        args, _ = parser.parse_known_args()
        return args

    return parser.parse_args()


def main() -> None:
    args = parse_args()
    summary_json = args.summary_json.resolve() if args.summary_json else resolve_latest_summary_json()

    print("JDH_ROOT         =", JDH_ROOT)
    print("summary_json     =", summary_json)
    print("reviews_root     =", args.reviews_root)
    print("local_qdrant_dir =", args.local_qdrant_dir)
    print("qdrant_mode      =", args.qdrant_mode)
    print("qdrant_url       =", args.qdrant_url)
    print("embed_model_id   =", EMBEDDING_MODEL_ID)
    print("top_k            =", args.top_k)
    print("output_dir       =", args.output_dir)

    prepared_reviews = prepare_reviews_payload(
        summary_json=summary_json,
        reviews_root=args.reviews_root,
        local_qdrant_dir=args.local_qdrant_dir,
        qdrant_mode=args.qdrant_mode,
        qdrant_url=args.qdrant_url,
        qdrant_api_key=args.qdrant_api_key,
        top_k=args.top_k,
    )

    if not prepared_reviews:
        raise RuntimeError("No prepared reviews were built.")

    paths = save_outputs(prepared_reviews, args.output_dir)

    print("\nDone.")
    print("Files written:")
    for key, path in paths.items():
        print(f" - {key}: {path.resolve()}")


if __name__ == "__main__":
    main()

# Grid5000

In [ ]:
%%bash
set -euo pipefail

APPLY=0
if [[ "${1:-}" == "--apply" ]]; then
  APPLY=1
fi

HOME_DIR="${HOME:?}"

say() {
  printf '%s\n' "$*"
}

run_cmd() {
  if [[ "$APPLY" -eq 1 ]]; then
    eval "$@"
  else
    say "[DRY-RUN] $*"
  fi
}

report_usage() {
  say
  say "===== SPACE REPORT ====="
  say "HOME: $HOME_DIR"
  say

  say "[df -hP \$HOME]"
  df -hP "$HOME_DIR" 2>/dev/null || true
  say

  say "[df -iP \$HOME]"
  df -iP "$HOME_DIR" 2>/dev/null || true
  say

  say "[quota]"
  (quota -s 2>/dev/null || quota -v 2>/dev/null || true)
  say

  say "[du -sh key paths]"
  du -sh "$HOME_DIR" 2>/dev/null || true
  [[ -d "$HOME_DIR/.qa_launcher" ]] && du -sh "$HOME_DIR/.qa_launcher" 2>/dev/null || true
  [[ -d "$HOME_DIR/model-store" ]] && du -sh "$HOME_DIR/model-store" 2>/dev/null || true
  [[ -d "$HOME_DIR/.cache" ]] && du -sh "$HOME_DIR/.cache" 2>/dev/null || true
  say

  say "[largest entries under HOME depth<=2]"
  du -xhd 2 "$HOME_DIR" 2>/dev/null | sort -h | tail -n 40 || true
  say "========================"
  say
}

delete_path_if_exists() {
  local p="$1"
  if [[ -e "$p" ]]; then
    run_cmd "rm -rf -- $(printf '%q' "$p")"
  fi
}

delete_matching_files() {
  local pattern="$1"
  if [[ "$APPLY" -eq 1 ]]; then
    find "$HOME_DIR" -maxdepth 1 -type f -name "$pattern" -print -delete 2>/dev/null || true
  else
    find "$HOME_DIR" -maxdepth 1 -type f -name "$pattern" -print 2>/dev/null | sed 's/^/[DRY-RUN] would delete /' || true
  fi
}

say "Mode: $([[ "$APPLY" -eq 1 ]] && echo APPLY || echo DRY-RUN)"
report_usage

say "Cleaning common quota-heavy artifacts in HOME..."

delete_path_if_exists "$HOME_DIR/.qa_launcher"
delete_matching_files 'qa_answer_*.txt'
delete_path_if_exists "$HOME_DIR/model-store"
delete_path_if_exists "$HOME_DIR/.cache/huggingface"
delete_path_if_exists "$HOME_DIR/.cache/pip"
delete_path_if_exists "$HOME_DIR/.cache/torch"
delete_path_if_exists "$HOME_DIR/.cache/xdg"
delete_path_if_exists "$HOME_DIR/.cache/uv"
delete_path_if_exists "$HOME_DIR/.cache/matplotlib"
delete_path_if_exists "$HOME_DIR/.cache/pypoetry"
delete_path_if_exists "$HOME_DIR/.local/share/Trash"
delete_path_if_exists "$HOME_DIR/.nv"
delete_path_if_exists "$HOME_DIR/.triton"
delete_path_if_exists "$HOME_DIR/.config/Code/Cache"
delete_path_if_exists "$HOME_DIR/.config/Code/CachedData"

if [[ "$APPLY" -eq 1 ]]; then
  find "$HOME_DIR/.cache" -type d -empty -delete 2>/dev/null || true
  find "$HOME_DIR/.local" -type d -empty -delete 2>/dev/null || true
  find "$HOME_DIR/.config" -type d -empty -delete 2>/dev/null || true
else
  say "[DRY-RUN] would prune empty dirs under ~/.cache ~/.local ~/.config"
fi

say
say "Cleaning done."
report_usage

In [ ]:
%%bash
cat > clean_my_grid5000_disk.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail

APPLY=0
if [[ "${1:-}" == "--apply" ]]; then
  APPLY=1
fi

HOME_DIR="${HOME:?}"

say() {
  printf '%s\n' "$*"
}

run_cmd() {
  if [[ "$APPLY" -eq 1 ]]; then
    eval "$@"
  else
    say "[DRY-RUN] $*"
  fi
}

report_usage() {
  say
  say "===== SPACE REPORT ====="
  say "HOME: $HOME_DIR"
  say

  say "[df -hP \$HOME]"
  df -hP "$HOME_DIR" 2>/dev/null || true
  say

  say "[df -iP \$HOME]"
  df -iP "$HOME_DIR" 2>/dev/null || true
  say

  say "[quota]"
  (quota -s 2>/dev/null || quota -v 2>/dev/null || true)
  say

  say "[du -sh key paths]"
  du -sh "$HOME_DIR" 2>/dev/null || true
  [[ -d "$HOME_DIR/.qa_launcher" ]] && du -sh "$HOME_DIR/.qa_launcher" 2>/dev/null || true
  [[ -d "$HOME_DIR/model-store" ]] && du -sh "$HOME_DIR/model-store" 2>/dev/null || true
  [[ -d "$HOME_DIR/.cache" ]] && du -sh "$HOME_DIR/.cache" 2>/dev/null || true
  say
}

delete_path_if_exists() {
  local p="$1"
  if [[ -e "$p" ]]; then
    run_cmd "rm -rf -- $(printf '%q' "$p")"
  fi
}

delete_matching_files() {
  local pattern="$1"
  if [[ "$APPLY" -eq 1 ]]; then
    find "$HOME_DIR" -maxdepth 1 -type f -name "$pattern" -print -delete 2>/dev/null || true
  else
    find "$HOME_DIR" -maxdepth 1 -type f -name "$pattern" -print 2>/dev/null | sed 's/^/[DRY-RUN] would delete /' || true
  fi
}

say "Mode: $([[ "$APPLY" -eq 1 ]] && echo APPLY || echo DRY-RUN)"
report_usage

delete_path_if_exists "$HOME_DIR/.qa_launcher"
delete_matching_files 'qa_answer_*.txt'
delete_path_if_exists "$HOME_DIR/model-store"
delete_path_if_exists "$HOME_DIR/.cache/huggingface"
delete_path_if_exists "$HOME_DIR/.cache/pip"
delete_path_if_exists "$HOME_DIR/.cache/torch"
delete_path_if_exists "$HOME_DIR/.cache/xdg"
delete_path_if_exists "$HOME_DIR/.cache/uv"
delete_path_if_exists "$HOME_DIR/.local/share/Trash"

say
say "Cleaning done."
report_usage
EOF

chmod +x clean_my_grid5000_disk.sh
./clean_my_grid5000_disk.sh

In [ ]:
#!/usr/bin/env python3
import datetime
import json
import re
import shlex
import subprocess
import textwrap
import time
from pathlib import Path


# =========================================================
# LOCAL JDH PATH RESOLUTION
# =========================================================
def resolve_local_jdh_root() -> Path:
    candidates = [
        Path("./"),
        Path("JDH"),
        Path("./JDH"),
        Path("/JDH"),
        Path.cwd() / "JDH",
        Path.home() / "JDH",
    ]

    seen = set()
    ordered = []
    for p in candidates:
        rp = p.resolve()
        if str(rp) not in seen:
            seen.add(str(rp))
            ordered.append(rp)

    for p in ordered:
        if p.exists() and p.is_dir():
            return p

    cwd_listing = []
    try:
        cwd_listing = sorted(x.name for x in Path.cwd().iterdir())
    except Exception:
        cwd_listing = []

    raise FileNotFoundError(
        "Could not find JDH folder.\n"
        + "Checked: "
        + ", ".join(str(p) for p in ordered)
        + "\nCurrent working directory: "
        + str(Path.cwd())
        + "\nFiles in cwd: "
        + ", ".join(cwd_listing)
    )


JDH_ROOT = resolve_local_jdh_root()


# =========================================================
# GRID5000 / MODEL SETTINGS
# =========================================================
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"
MODEL_REVISION = "main"
TRUST_REMOTE_CODE = "1"

# 32B BF16 should run on 2 x 64GB MI210 with device_map="auto".
GPU_REQUEST = 2
EXPECTED_GPU_COUNT = 2

WALLTIME = "10:00:00"
MAX_WAIT_MINUTES = 600
POLL_SECONDS = 25
HOST_PREDICATE = "network_address like 'compute-node-%'"

# MI210 / ROCm.
HSA_OVERRIDE_GFX_VERSION = "9.0.10"

# Stable ROCm stack you already used successfully.
TORCH_INDEX_URL = "https://download.pytorch.org/whl/rocm6.1"
TORCH_VERSION = "2.5.1"
TORCHVISION_VERSION = "0.20.1"
TORCHAUDIO_VERSION = "2.5.1"

SSH_HOST = "lux"
LOCAL_RESULTS_ROOT = JDH_ROOT / "g5k_review_audit_results"
PREPARED_REVIEWS_DIR = JDH_ROOT / "prepared_audit_prompts"


def resolve_prepared_reviews_file() -> Path:
    candidates = [
        PREPARED_REVIEWS_DIR / "prepared_reviews.json",
        PREPARED_REVIEWS_DIR / "prepared_reviews.jsonl",
        PREPARED_REVIEWS_DIR / "comment_prompts.json",
        PREPARED_REVIEWS_DIR / "comment_prompts.jsonl",
    ]

    for p in candidates:
        if p.exists() and p.is_file():
            return p

    available = []
    if PREPARED_REVIEWS_DIR.exists():
        try:
            available = sorted(x.name for x in PREPARED_REVIEWS_DIR.iterdir())
        except Exception:
            available = []

    raise FileNotFoundError(
        "Could not find prepared reviews file under "
        + str(PREPARED_REVIEWS_DIR.resolve())
        + ". Looked for: "
        + ", ".join(str(p.name) for p in candidates)
        + ". Available files: "
        + ", ".join(available)
    )


def load_prepared_reviews_file(path: Path):
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError("Prepared reviews file is empty: " + str(path.resolve()))

    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for key in ("prepared_reviews", "reviews", "items", "data"):
                value = data.get(key)
                if isinstance(value, list):
                    return value
            raise ValueError(
                "Prepared reviews JSON root is a dict but no list was found under keys "
                + "prepared_reviews/reviews/items/data in "
                + str(path.resolve())
            )
    except json.JSONDecodeError:
        pass

    rows = []
    for lineno, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except Exception as exc:
            raise ValueError(
                "Could not parse prepared reviews as JSON or JSONL at line "
                + str(lineno)
                + " in "
                + str(path.resolve())
                + ": "
                + str(exc)
            ) from exc

    if rows:
        return rows

    raise ValueError("Could not parse prepared reviews file: " + str(path.resolve()))


class Grid5000Client:
    def __init__(self, ssh_host: str = SSH_HOST, local_results_root: Path | str = LOCAL_RESULTS_ROOT):
        self.ssh_host = ssh_host
        self.local_results_root = Path(local_results_root)
        self.local_results_root.mkdir(parents=True, exist_ok=True)

        self.remote_user = None
        self.frontend_host = None
        self.base_dir = None

    def run(self, cmd: str, input_text: str | None = None) -> str:
        cmd = textwrap.dedent(cmd).strip()
        if not cmd:
            return ""

        remote = "bash -lc " + shlex.quote(cmd)
        p = subprocess.run(
            ["ssh", self.ssh_host, remote],
            input=input_text,
            text=True,
            capture_output=True,
        )
        if p.returncode != 0:
            raise RuntimeError(
                "Remote cmd failed rc="
                + str(p.returncode)
                + "\n--- CMD ---\n"
                + cmd
                + "\n--- STDERR ---\n"
                + p.stderr
            )
        return p.stdout

    def run_ok(self, cmd: str) -> str:
        try:
            return self.run(cmd)
        except RuntimeError:
            return ""

    def space_report(self, remote_path: str) -> str:
        cmd = "\n".join([
            "set -u",
            "TARGET=" + shlex.quote(remote_path),
            'if [ -d "$TARGET" ]; then CHECK="$TARGET"; else CHECK="$(dirname "$TARGET")"; fi',
            'while [ ! -e "$CHECK" ] && [ "$CHECK" != "/" ]; do CHECK="$(dirname "$CHECK")"; done',
            'echo "check_path=$CHECK"',
            "echo",
            'echo "[df -hP]"',
            'df -hP "$CHECK" 2>/dev/null || true',
            "echo",
            'echo "[df -iP]"',
            'df -iP "$CHECK" 2>/dev/null || true',
            "echo",
            'echo "[du -sh]"',
            'du -sh "$CHECK" 2>/dev/null || true',
            'test -d "$HOME/.qa_launcher" && du -sh "$HOME/.qa_launcher" 2>/dev/null || true',
            'test -d "$HOME/model-store" && du -sh "$HOME/model-store" 2>/dev/null || true',
            "echo",
            'echo "[quota]"',
            "(quota -s 2>/dev/null || quota -v 2>/dev/null || true)",
        ])
        return self.run_ok(cmd)

    def ensure_identity(self) -> None:
        if self.remote_user is None:
            self.remote_user = self.run("id -un").strip()
        if self.frontend_host is None:
            self.frontend_host = self.run("hostname -f 2>/dev/null || hostname").strip()

    def prepare_home(self) -> str:
        cmd = "\n".join([
            "set -euo pipefail",
            'rm -rf "$HOME/.qa_launcher" 2>/dev/null || true',
            'mkdir -p "$HOME/.qa_launcher"',
            "find \"$HOME\" -maxdepth 1 -type f -name 'qa_answer_*.txt' -delete 2>/dev/null || true",
            'rm -rf "$HOME/model-store" 2>/dev/null || true',
            'rm -rf "$HOME/.cache/huggingface" 2>/dev/null || true',
            'rm -rf "$HOME/.cache/pip" 2>/dev/null || true',
            'rm -rf "$HOME/.cache/torch" 2>/dev/null || true',
            'rm -rf "$HOME/.cache/xdg" 2>/dev/null || true',
            'rm -rf "$HOME/.cache/uv" 2>/dev/null || true',
            'rm -rf "$HOME/.local/share/Trash" 2>/dev/null || true',
            'echo "remote HOME prep done"',
        ])
        return self.run(cmd)

    def resolve_base_dir(self) -> str:
        self.ensure_identity()
        self.base_dir = self.run("echo $HOME").strip()

        print("[REMOTE] preparing HOME...", flush=True)
        try:
            out = self.prepare_home().strip()
            if out:
                print(out, flush=True)
        except Exception as exc:
            report = self.space_report(self.base_dir)
            raise RuntimeError(str(exc) + "\n\n[REMOTE] HOME free space / quota report:\n" + report)

        print("[REMOTE] using HOME:", self.base_dir, flush=True)
        print("[REMOTE] free space / quota report:", flush=True)
        print(self.space_report(self.base_dir).strip(), flush=True)
        return self.base_dir

    def make_workdir(self) -> str:
        if self.base_dir is None:
            self.resolve_base_dir()

        cmd = "\n".join([
            'mkdir -p "$HOME/.qa_launcher"',
            'mktemp -d -p "$HOME/.qa_launcher" qa_run.XXXXXX',
        ])
        return self.run(cmd).strip()

    def upload_text(self, remote_dir: str, filename: str, content: str) -> str:
        remote_path = remote_dir + "/" + filename
        cmd = (
            "set -euo pipefail\n"
            + "cat > "
            + shlex.quote(remote_path)
            + "\n"
            + "chmod 600 "
            + shlex.quote(remote_path)
        )
        self.run(cmd, input_text=content)
        return remote_path

    def mkdir_p(self, remote_path: str) -> None:
        self.run("mkdir -p " + shlex.quote(remote_path))

    def upload_file(self, local_path: Path, remote_path: str) -> str:
        local_path = Path(local_path)
        if not local_path.exists():
            raise FileNotFoundError("Missing local file: " + str(local_path.resolve()))

        remote_parent = str(Path(remote_path).parent)
        self.mkdir_p(remote_parent)

        print(
            "[UPLOAD] local="
            + str(local_path.resolve())
            + " -> remote="
            + remote_path,
            flush=True,
        )

        tmp_remote = remote_path + ".part.$$"
        remote_cmd = "\n".join([
            "set -euo pipefail",
            "mkdir -p " + shlex.quote(remote_parent),
            "TMP_REMOTE=" + shlex.quote(tmp_remote),
            "trap 'rm -f \"$TMP_REMOTE\"' EXIT",
            'cat > "$TMP_REMOTE"',
            "mv -f \"$TMP_REMOTE\" " + shlex.quote(remote_path),
            "chmod 600 " + shlex.quote(remote_path),
            "ls -lh " + shlex.quote(remote_path),
            "trap - EXIT",
        ])

        p = subprocess.run(
            ["ssh", self.ssh_host, "bash -lc " + shlex.quote(remote_cmd)],
            input=local_path.read_bytes(),
            capture_output=True,
        )

        if p.returncode != 0:
            stdout = (p.stdout or b"").decode("utf-8", errors="replace")
            stderr = (p.stderr or b"").decode("utf-8", errors="replace")
            raise RuntimeError(
                "Failed to upload file.\n"
                + "local_path="
                + str(local_path.resolve())
                + "\nremote_path="
                + remote_path
                + "\n\n--- SSH STDOUT ---\n"
                + stdout
                + "\n--- SSH STDERR ---\n"
                + stderr
            )

        stdout = (p.stdout or b"").decode("utf-8", errors="replace").strip()
        if stdout:
            print(stdout, flush=True)

        return remote_path

    def write_remote_script(self, remote_dir: str, script_name: str, script_text: str) -> str:
        remote_path = remote_dir + "/" + script_name
        self.run(
            "cat > "
            + shlex.quote(remote_path)
            + " <<'EOF'\n"
            + script_text
            + "\nEOF\nchmod 700 "
            + shlex.quote(remote_path)
        )
        return remote_path

    def submit_job(self, remote_dir: str, script_path: str, gpu_request: int, walltime: str, host_predicate: str) -> tuple[str, str]:
        out = self.run(
            "oarsub -l "
            + shlex.quote("host=1/gpu=" + str(gpu_request) + ",walltime=" + walltime)
            + " -p "
            + shlex.quote(host_predicate)
            + " -d "
            + shlex.quote(remote_dir)
            + " -O "
            + shlex.quote(remote_dir + "/stdout.%jobid%.txt")
            + " -E "
            + shlex.quote(remote_dir + "/stderr.%jobid%.txt")
            + " "
            + shlex.quote(script_path)
        )

        m = re.search(r"OAR_JOB_ID=(\d+)", out)
        if not m:
            raise RuntimeError("Could not parse OAR_JOB_ID.\n--- oarsub output ---\n" + out)
        return m.group(1), out

    def oar_field(self, jobid: str, field: str) -> str:
        field = re.sub(r"[^A-Za-z0-9_]", "", field)
        return self.run_ok(
            "oarstat -j "
            + shlex.quote(str(jobid))
            + " -f 2>/dev/null | "
            + "sed -n 's/^[[:space:]]*"
            + field
            + "[[:space:]]*=[[:space:]]*//p' | "
            + "head -n 1 || true"
        ).strip()

    def job_state(self, jobid: str) -> str:
        return self.oar_field(jobid, "state") or "UNKNOWN"

    def job_host(self, jobid: str) -> str:
        return self.oar_field(jobid, "assigned_hostnames")

    def remote_exists_nonempty(self, path: str) -> bool:
        out = self.run_ok("test -s " + shlex.quote(path) + " && echo YES || true").strip()
        return out == "YES"

    def read_remote_text(self, path: str) -> str:
        return self.run_ok("cat " + shlex.quote(path) + " 2>/dev/null || true")

    def wait_for_answer(
        self,
        jobid: str,
        remote_dir: str,
        answer_path: str,
        status_path: str,
        poll_seconds: int,
        max_wait_minutes: int,
    ) -> tuple[str, str]:
        deadline = time.time() + max_wait_minutes * 60
        final_state = "UNKNOWN"
        last_live_len = 0
        last_live_text = ""

        while time.time() < deadline:
            state = self.job_state(jobid)
            host = self.job_host(jobid)
            ts = datetime.datetime.now().strftime("%H:%M:%S")
            print("[" + ts + "] state=" + state + (" host=" + host if host else ""), flush=True)
            final_state = state

            live_log_path = remote_dir + "/live.log"
            live_text = self.read_remote_text(live_log_path)
            if len(live_text) < last_live_len:
                last_live_len = 0
            if len(live_text) > last_live_len:
                new_live = live_text[last_live_len:]
                if new_live.strip():
                    print("\n[live.log]")
                    print(new_live, end="" if new_live.endswith("\n") else "\n", flush=True)
                last_live_len = len(live_text)
                last_live_text = live_text

            if self.remote_exists_nonempty(answer_path):
                answer = self.read_remote_text(answer_path)
                if answer.strip():
                    return state, answer

            if state in ("Terminated", "Error", "Finishing"):
                answer = self.read_remote_text(answer_path)
                if answer.strip():
                    return state, answer
                status_text = self.read_remote_text(status_path)
                if status_text.strip():
                    return state, status_text

            time.sleep(poll_seconds)

        status_text = self.read_remote_text(status_path)
        if status_text.strip():
            return final_state, status_text

        return final_state, last_live_text

    def copy_remote_dir_to_local(self, remote_dir: str, local_dir: Path) -> Path:
        local_dir.mkdir(parents=True, exist_ok=True)
        rsync_cmd = [
            "rsync",
            "-az",
            "-e",
            "ssh",
            self.ssh_host + ":" + remote_dir + "/",
            str(local_dir) + "/",
        ]
        p = subprocess.run(rsync_cmd, text=True, capture_output=True)
        if p.returncode != 0:
            raise RuntimeError(
                "Failed to copy remote directory.\n"
                + "remote_dir="
                + remote_dir
                + "\nlocal_dir="
                + str(local_dir)
                + "\n"
                + p.stderr
            )
        return local_dir

    def cleanup_remote_job_artifacts(self, remote_workdir: str, jobid: str) -> None:
        cmd = "\n".join([
            "set -euo pipefail",
            "rm -rf " + shlex.quote(remote_workdir) + " 2>/dev/null || true",
            'rm -f "$HOME/qa_answer_' + str(jobid) + '.txt" 2>/dev/null || true',
            'rmdir "$HOME/.qa_launcher" 2>/dev/null || true',
            'echo "remote cleanup done"',
        ])
        out = self.run_ok(cmd).strip()
        if out:
            print(out, flush=True)


def main():
    prepared_reviews_json = resolve_prepared_reviews_file()
    prepared_reviews_preview = load_prepared_reviews_file(prepared_reviews_json)

    print("cwd                    =", Path.cwd())
    print("jdh_root               =", JDH_ROOT)
    print("prepared_reviews_json  =", prepared_reviews_json)
    print("prepared_reviews_count =", len(prepared_reviews_preview))
    print("model_id               =", MODEL_ID)
    print("model_revision         =", MODEL_REVISION)
    print("gpu_request            =", GPU_REQUEST)
    print("walltime               =", WALLTIME)

    g5k = Grid5000Client()

    remote_base = g5k.resolve_base_dir()
    print("\nResolved remote base dir:", remote_base)

    remote_workdir = g5k.make_workdir()
    print("\nRemote workdir:", remote_workdir)

    remote_outputs_dir = remote_workdir + "/outputs"
    remote_reports_dir = remote_outputs_dir + "/audit_reports"

    g5k.mkdir_p(remote_outputs_dir)
    g5k.mkdir_p(remote_reports_dir)

    g5k.upload_file(prepared_reviews_json, remote_workdir + "/prepared_reviews.json")
    g5k.upload_text(remote_workdir, "model.txt", MODEL_ID)
    g5k.upload_text(remote_workdir, "model_revision.txt", MODEL_REVISION)
    g5k.upload_text(remote_workdir, "trust_remote_code.txt", TRUST_REMOTE_CODE)
    g5k.upload_text(remote_workdir, "gpu_request.txt", str(GPU_REQUEST))
    g5k.upload_text(remote_workdir, "hsa_override_gfx_version.txt", HSA_OVERRIDE_GFX_VERSION)
    g5k.upload_text(remote_workdir, "torch_index_url.txt", TORCH_INDEX_URL)
    g5k.upload_text(remote_workdir, "torch_version.txt", TORCH_VERSION)
    g5k.upload_text(remote_workdir, "torchvision_version.txt", TORCHVISION_VERSION)
    g5k.upload_text(remote_workdir, "torchaudio_version.txt", TORCHAUDIO_VERSION)

    run_sh = r'''#!/usr/bin/env bash
set -euo pipefail

WORKDIR="$PWD"
OUTPUTS_DIR="$WORKDIR/outputs"
REPORTS_DIR="$OUTPUTS_DIR/audit_reports"
mkdir -p "$REPORTS_DIR"

ANSWER_FILE="$WORKDIR/answer.txt"
STATUS_FILE="$WORKDIR/status.txt"
SUMMARY_FILE="$WORKDIR/summary.json"
LIVE_LOG="$WORKDIR/live.log"
: > "$LIVE_LOG"

exec > >(tee -a "$LIVE_LOG") 2>&1

echo "BOOTSTRAP_STARTED" > "$STATUS_FILE"

export MODEL_ID="$(cat "$WORKDIR/model.txt")"
export MODEL_REVISION="$(cat "$WORKDIR/model_revision.txt")"
export TRUST_REMOTE_CODE_VALUE="$(cat "$WORKDIR/trust_remote_code.txt")"
export GPU_REQUEST_VALUE="$(cat "$WORKDIR/gpu_request.txt")"
export HSA_OVERRIDE_GFX_VERSION_VALUE="$(cat "$WORKDIR/hsa_override_gfx_version.txt")"
export TORCH_INDEX_URL="$(cat "$WORKDIR/torch_index_url.txt")"
export TORCH_VERSION="$(cat "$WORKDIR/torch_version.txt")"
export TORCHVISION_VERSION="$(cat "$WORKDIR/torchvision_version.txt")"
export TORCHAUDIO_VERSION="$(cat "$WORKDIR/torchaudio_version.txt")"

export JOB_TMP="/tmp/$USER-job-$OAR_JOB_ID"
export VENV="/tmp/$USER-venv-$OAR_JOB_ID"

cleanup() {
  rc=$?

  if [ "$rc" -ne 0 ]; then
    echo "FAILED rc=$rc" > "$STATUS_FILE"
  fi

  echo
  echo "[CLEANUP] removing tmp artifacts..."
  rm -rf "$JOB_TMP" "$VENV" 2>/dev/null || true

  echo
  echo "[CLEANUP] tmp after removal"
  df -hP /tmp || true
  df -iP /tmp || true
}
trap cleanup EXIT

rm -rf "$JOB_TMP" "$VENV"
mkdir -p "$JOB_TMP"

export HF_HOME="$JOB_TMP/hf-home"
export HF_HUB_CACHE="$HF_HOME/hub"
export XDG_CACHE_HOME="$JOB_TMP/xdg"
export PIP_CACHE_DIR="$JOB_TMP/pip"
export MODEL_LOAD_DIR="$JOB_TMP/model-repo"
export OFFLOAD_DIR="$JOB_TMP/offload"

mkdir -p \
  "$HF_HUB_CACHE" \
  "$XDG_CACHE_HOME" \
  "$PIP_CACHE_DIR" \
  "$MODEL_LOAD_DIR" \
  "$OFFLOAD_DIR"

echo
echo "[TMP df -hP]"
df -hP /tmp || true
echo
echo "[TMP df -iP]"
df -iP /tmp || true
echo
echo "[TMP du -sh]"
du -sh "$JOB_TMP" 2>/dev/null || true
echo
echo "[HOME quota]"
(quota -s 2>/dev/null || quota -v 2>/dev/null || true)
echo

GPU_LIST="$(python3 - <<'PYGPU'
import os
n = int(os.environ.get("GPU_REQUEST_VALUE", "2"))
print(",".join(str(i) for i in range(n)))
PYGPU
)"

export HIP_VISIBLE_DEVICES="$GPU_LIST"
export ROCR_VISIBLE_DEVICES="$GPU_LIST"
export CUDA_VISIBLE_DEVICES="$GPU_LIST"
export HSA_OVERRIDE_GFX_VERSION="$HSA_OVERRIDE_GFX_VERSION_VALUE"
export TOKENIZERS_PARALLELISM=false
export PYTHONUNBUFFERED=1
export PYTHONFAULTHANDLER=1
export PYTORCH_HIP_ALLOC_CONF="max_split_size_mb:128"
export NCCL_DEBUG=WARN

PYTHON_BIN=""
for candidate in python3.12 python3.11 python3.10; do
  if command -v "$candidate" >/dev/null 2>&1; then
    if "$candidate" - <<'PYVER'
import sys
raise SystemExit(0 if sys.version_info >= (3, 10) else 1)
PYVER
    then
      PYTHON_BIN="$candidate"
      break
    fi
  fi
done

if [ -n "$PYTHON_BIN" ]; then
  echo "[PYTHON] using system $PYTHON_BIN: $($PYTHON_BIN --version 2>&1)"
  "$PYTHON_BIN" -m venv "$VENV"
  source "$VENV/bin/activate"
else
  echo "[PYTHON] no system Python >= 3.10 found; installing temporary Miniforge Python 3.11"
  MINIFORGE_DIR="$JOB_TMP/miniforge"
  CONDA_ENV_PREFIX="$JOB_TMP/conda-env"
  MINIFORGE_SH="$JOB_TMP/miniforge.sh"

  if command -v curl >/dev/null 2>&1; then
    curl -L -o "$MINIFORGE_SH" "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"
  elif command -v wget >/dev/null 2>&1; then
    wget -O "$MINIFORGE_SH" "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh"
  else
    echo "ERROR: neither curl nor wget is available to install Miniforge" >&2
    exit 1
  fi

  bash "$MINIFORGE_SH" -b -p "$MINIFORGE_DIR"
  "$MINIFORGE_DIR/bin/conda" create -y -p "$CONDA_ENV_PREFIX" python=3.11 pip
  source "$MINIFORGE_DIR/etc/profile.d/conda.sh"
  conda activate "$CONDA_ENV_PREFIX"
fi

echo "[PYTHON] final interpreter: $(command -v python)"
python --version

python -m pip -q install -U pip wheel setuptools packaging accelerate

python -m pip -q install --no-cache-dir \
  "torch==${TORCH_VERSION}" \
  "torchvision==${TORCHVISION_VERSION}" \
  "torchaudio==${TORCHAUDIO_VERSION}" \
  --index-url "${TORCH_INDEX_URL}"

python -m pip -q install --no-cache-dir \
  "huggingface_hub>=0.23" \
  "safetensors>=0.4.3" \
  "transformers>=4.46.0" \
  "accelerate>=0.33.0" \
  "sentencepiece" \
  "protobuf" \
  "einops" \
  "tiktoken" \
  "tqdm"

python - <<'PYENV'
import sys
import torch
import transformers
print("PYTHON=", sys.version)
print("TORCH=", torch.__version__)
print("TRANSFORMERS=", transformers.__version__)
print("CUDA_AVAILABLE=", torch.cuda.is_available())
print("DEVICE_COUNT=", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("DEVICE", i, torch.cuda.get_device_name(i))
PYENV

python - <<'PYDL'
import os
from huggingface_hub import snapshot_download

token = os.environ.get("HF_TOKEN") or None

path = snapshot_download(
    repo_id=os.environ["MODEL_ID"],
    revision=os.environ.get("MODEL_REVISION", "main"),
    local_dir=os.environ["MODEL_LOAD_DIR"],
    token=token,
    resume_download=True,
)
print("HF model cached in TMP at:", path)
PYDL

echo "MODEL_LOAD_DIR=$MODEL_LOAD_DIR"
du -sh "$MODEL_LOAD_DIR" 2>/dev/null || true

cat > "$WORKDIR/audit_runner.py" <<'PYAUDIT'
from pathlib import Path
import gc
import json
import os
import re
import traceback
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

workdir = Path.cwd()
outputs_dir = workdir / "outputs"
reports_dir = outputs_dir / "audit_reports"
reports_dir.mkdir(parents=True, exist_ok=True)

answer_file = workdir / "answer.txt"
status_file = workdir / "status.txt"
summary_file = workdir / "summary.json"

MODEL_LOAD_DIR = os.environ["MODEL_LOAD_DIR"]
MODEL_ID = os.environ["MODEL_ID"]
OFFLOAD_DIR = os.environ["OFFLOAD_DIR"]

WEIGHTS = {
    "retrieval_strength": 0.15,
    "model_support_strength": 0.25,
    "evidence_specificity": 0.45,
    "section_coverage": 0.15,
}

SCORE_SCHEMA_HINT = {
    "reviewer_id": "string",
    "given_score_raw": "string",
    "normalized_score_0_to_1": 0.0,
    "score_alignment": "well_aligned | mostly_aligned | weakly_aligned | not_aligned",
    "score_confidence": 0.0,
    "score_justification": "string",
}


def print_flush(*args):
    print(*args, flush=True)


def atomic_write(path: Path, content: str) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(content, encoding="utf-8")
    tmp.replace(path)


def atomic_json(path: Path, obj) -> None:
    atomic_write(path, json.dumps(obj, indent=2, ensure_ascii=False) + "\n")


def clean_text(text) -> str:
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", " ")

    # Fix byte-level BPE artefacts sometimes visible in bad decodes.
    text = text.replace("\u0120", " ")
    text = text.replace("\u010A", "\n")

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def normalize_section_name(section) -> str:
    s = clean_text(section).lower()
    if not s or s in {"unknown", "none", "null", "n/a", "na"}:
        return ""
    return s


def safe_int(x, default=0) -> int:
    try:
        if x is None or x == "":
            return int(default)
        return int(x)
    except Exception:
        return int(default)


def safe_float(x, default=0.0) -> float:
    try:
        if x is None or x == "":
            return float(default)
        if isinstance(x, bool):
            return 1.0 if x else 0.0
        return float(x)
    except Exception:
        return float(default)


def clamp01(x, default=0.0) -> float:
    return max(0.0, min(1.0, safe_float(x, default)))


def safe_metric_float(x, default=0.0) -> float:
    if isinstance(x, str):
        mapping = {
            "very low": 0.1,
            "low": 0.25,
            "medium": 0.5,
            "moderate": 0.5,
            "high": 0.75,
            "very high": 0.9,
        }
        s = x.strip().lower()
        if s in mapping:
            return mapping[s]
    if isinstance(x, list):
        if not x:
            return float(default)
        return max(0.0, min(1.0, len(x) / 3.0))
    if isinstance(x, dict):
        return float(default)
    return clamp01(x, default)


def compute_retrieval_strength_from_chunks(retrieved_chunks: list[dict]) -> float:
    scores = []
    for c in retrieved_chunks or []:
        if not isinstance(c, dict):
            continue
        score = safe_float(c.get("retrieval_score", 0.0), 0.0)
        if score > 0:
            scores.append(clamp01(score))

    if not scores:
        return 0.0

    return round(sum(scores) / len(scores), 4)


def build_total_sections_by_paper(prepared_reviews: list[dict]) -> dict:
    sections_by_paper = {}

    for review in prepared_reviews or []:
        if not isinstance(review, dict):
            continue

        paper_id = str(review.get("paper_id", "")).strip()
        if not paper_id:
            continue

        sections_by_paper.setdefault(paper_id, set())

        for task in review.get("comment_tasks", []) or []:
            if not isinstance(task, dict):
                continue

            for chunk in task.get("retrieved_chunks", []) or []:
                if not isinstance(chunk, dict):
                    continue

                section = normalize_section_name(chunk.get("section", ""))
                if section:
                    sections_by_paper[paper_id].add(section)

    return {
        paper_id: max(1, len(sections))
        for paper_id, sections in sections_by_paper.items()
    }


def compute_section_coverage_from_retrieved_chunks(retrieved_chunks: list[dict], total_paper_sections: int) -> float:
    retrieved_sections = set()

    for c in retrieved_chunks or []:
        if not isinstance(c, dict):
            continue

        section = normalize_section_name(c.get("section", ""))
        if section:
            retrieved_sections.add(section)

    if not retrieved_sections:
        return 0.0

    denominator = max(1, int(total_paper_sections))
    return round(max(0.0, min(1.0, len(retrieved_sections) / denominator)), 4)


def score_to_unit_interval(score_raw: str) -> float:
    s = clean_text(score_raw).lower()
    m = re.match(r"^([0-9]+(?:\.[0-9]+)?)\s*/\s*([0-9]+(?:\.[0-9]+)?)$", s)
    if m:
        num, den = float(m.group(1)), float(m.group(2))
        if den > 0:
            return max(0.0, min(1.0, num / den))
    m = re.match(r"^([0-9]+(?:\.[0-9]+)?)$", s)
    if m:
        val = float(m.group(1))
        if 0 <= val <= 1:
            return val
        if 0 <= val <= 10:
            return val / 10.0
    mapping = {
        "strong reject": 0.10,
        "reject": 0.25,
        "weak reject": 0.40,
        "borderline": 0.50,
        "weak accept": 0.65,
        "accept": 0.80,
        "strong accept": 0.95,
    }
    return mapping.get(s, 0.50)


def compute_blended_confidence(audit: dict) -> float:
    blended = (
        WEIGHTS["retrieval_strength"] * clamp01(audit.get("retrieval_strength", 0.0))
        + WEIGHTS["model_support_strength"] * clamp01(audit.get("confidence", 0.0))
        + WEIGHTS["evidence_specificity"] * clamp01(audit.get("evidence_specificity", 0.0))
        + WEIGHTS["section_coverage"] * clamp01(audit.get("section_coverage", 0.0))
    )
    return round(max(0.0, min(1.0, blended)), 4)


def confidence_band(x: float) -> str:
    if x >= 0.85:
        return "very_high"
    if x >= 0.70:
        return "high"
    if x >= 0.50:
        return "medium"
    return "low"


def parse_json_like(text):
    return json.loads(text)


def extract_json_block(text: str) -> str:
    if text is None:
        raise ValueError("Empty model output.")

    raw = clean_text(text)

    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL | re.IGNORECASE)
    raw = raw.replace("<think>", "").replace("</think>", "")
    raw = raw.replace("<|channel|>final<|message|>", "")
    raw = raw.replace("<|end|>", "")
    raw = raw.replace("<|return|>", "")
    raw = raw.replace("<|im_end|>", "")
    raw = raw.strip()

    if not raw:
        raise ValueError("Empty model output.")

    candidates = []

    fence_match = re.search(r"```(?:json)?\s*(.*?)\s*```", raw, flags=re.DOTALL | re.IGNORECASE)
    if fence_match:
        candidates.append(fence_match.group(1).strip())

    candidates.append(raw)

    start_positions = [i for i, ch in enumerate(raw) if ch in "[{"]

    for start in start_positions:
        opener = raw[start]
        closer = "}" if opener == "{" else "]"
        depth = 0
        in_string = False
        escape = False

        for i in range(start, len(raw)):
            ch = raw[i]

            if in_string:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_string = False
                continue

            if ch == '"':
                in_string = True
            elif ch == opener:
                depth += 1
            elif ch == closer:
                depth -= 1
                if depth == 0:
                    candidates.append(raw[start:i + 1])
                    break

    seen = set()
    for candidate in candidates:
        candidate = candidate.strip()
        if not candidate or candidate in seen:
            continue
        seen.add(candidate)

        try:
            parse_json_like(candidate)
            return candidate
        except Exception:
            pass

    raise ValueError("No JSON block found in model output.\n\nRAW MODEL OUTPUT:\n" + raw[:4000])


def load_prepared_reviews(path: Path):
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError("Prepared reviews file is empty: " + str(path))

    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            for key in ("prepared_reviews", "reviews", "items", "data"):
                value = data.get(key)
                if isinstance(value, list):
                    return value
            raise ValueError("Prepared reviews JSON root is a dict but no list was found.")
    except json.JSONDecodeError:
        pass

    rows = []
    for lineno, line in enumerate(text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        rows.append(json.loads(line))
    return rows


def first_model_device(model):
    for p in model.parameters():
        return p.device
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def build_model():
    print_flush("Loading tokenizer:", MODEL_LOAD_DIR)
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_LOAD_DIR,
        trust_remote_code=True,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    gpu_count = torch.cuda.device_count()
    max_memory = {}
    for i in range(gpu_count):
        max_memory[i] = "58GiB"
    max_memory["cpu"] = "180GiB"

    print_flush("Loading model:", MODEL_LOAD_DIR)
    print_flush("GPU count:", gpu_count)
    print_flush("max_memory:", max_memory)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_LOAD_DIR,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
        max_memory=max_memory,
        offload_folder=OFFLOAD_DIR,
    )

    model.eval()
    print_flush("Model loaded.")
    print_flush("First model device:", first_model_device(model))
    return tokenizer, model


def compact_chunk_for_prompt(c: dict) -> dict:
    return {
        "chunk_id": c.get("chunk_id", ""),
        "section": c.get("section", "unknown"),
        "page_start": safe_int(c.get("page_start"), 0),
        "page_end": safe_int(c.get("page_end"), 0),
        "retrieval_score": safe_float(c.get("retrieval_score", 0.0), 0.0),
        "full_chunk_text": clean_text(c.get("full_chunk_text", ""))[:1800],
    }


def build_comment_messages(task: dict) -> list[dict]:
    payload = {
        "comment_id": str(task.get("comment_id", "")),
        "question": clean_text(task.get("question", "")),
        "reviewer_comment": clean_text(task.get("reviewer_comment", "")),
        "retrieved_chunks": [
            compact_chunk_for_prompt(c)
            for c in (task.get("retrieved_chunks", []) or [])
            if isinstance(c, dict)
        ],
    }

    return [
        {
            "role": "system",
            "content": (
                "You are a strict JSON-only academic evidence auditor. "
                "Use only the retrieved chunks. "
                "Return one valid JSON object only. "
                "No markdown. No code fence. No preamble. "
                "The first character must be { and the last character must be }. "
                "Use exactly these keys: "
                "comment_id, support_label, confidence, evidence, reasoning_summary, outcome, evidence_specificity. "
                "support_label must be one of: supported, partially_supported, not_supported, insufficient_evidence. "
                "confidence must be a number between 0 and 1. "
                "evidence_specificity must be a number between 0 and 1. "
                "evidence must contain at most two objects. "
                "Each evidence object must have: chunk_id, section, page_start, page_end, snippet. "
                "Each snippet must be under 35 words."
            ),
        },
        {
            "role": "user",
            "content": json.dumps(payload, ensure_ascii=False),
        },
    ]


def build_score_messages(review_payload: dict, comment_audits: list[dict]) -> list[dict]:
    compact_comment_audits = []
    for c in comment_audits:
        compact_comment_audits.append(
            {
                "comment_id": c["comment_id"],
                "question": c.get("question", ""),
                "reviewer_comment": c["reviewer_comment"],
                "support_label": c["support_label"],
                "confidence": c["confidence"],
                "retrieval_strength": c["retrieval_strength"],
                "evidence_specificity": c["evidence_specificity"],
                "section_coverage": c["section_coverage"],
                "reasoning_summary": c["reasoning_summary"],
                "outcome": c.get("outcome", ""),
            }
        )

    payload = {
        "reviewer_id": review_payload["reviewer_id"],
        "given_score_raw": review_payload.get("score_raw", ""),
        "normalized_score_0_to_1": score_to_unit_interval(review_payload.get("score_raw", "")),
        "comment_audits": compact_comment_audits,
        "required_json_shape": SCORE_SCHEMA_HINT,
    }

    return [
        {
            "role": "system",
            "content": (
                "You are a strict JSON-only score alignment auditor. "
                "Return one valid JSON object only. "
                "No markdown. No code fence. No preamble. "
                "The first character must be { and the last character must be }. "
                "Use exactly these keys: reviewer_id, given_score_raw, normalized_score_0_to_1, "
                "score_alignment, score_confidence, score_justification. "
                "score_alignment must be one of: well_aligned, mostly_aligned, weakly_aligned, not_aligned."
            ),
        },
        {
            "role": "user",
            "content": json.dumps(payload, ensure_ascii=False),
        },
    ]


@torch.no_grad()
def model_generate_json(tokenizer, model, messages, max_input_tokens: int = 4096, max_new_tokens: int = 384):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # JSON prefill. The decoded answer below is prefixed with "{" too.
    prompt = prompt + "\n{"

    encoded = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens,
    )

    device = first_model_device(model)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    input_len = encoded["input_ids"].shape[-1]

    generated = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    new_tokens = generated[0][input_len:]
    raw_tail = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    raw_answer = clean_text("{" + raw_tail)

    debug_dir = outputs_dir / "debug_raw_generations"
    debug_dir.mkdir(parents=True, exist_ok=True)
    raw_path = debug_dir / ("raw_" + str(int(time.time() * 1000)) + ".txt")
    raw_path.write_text(raw_answer, encoding="utf-8")

    block = extract_json_block(raw_answer)
    data = parse_json_like(block)

    if isinstance(data, list):
        dict_items = [x for x in data if isinstance(x, dict)]
        if dict_items:
            data = dict_items[0]
        else:
            raise ValueError("Model output is a JSON list, but it contains no JSON object.")

    if not isinstance(data, dict):
        raise ValueError("Model output is not a JSON object.\n\nRAW MODEL OUTPUT:\n" + raw_answer[:4000])

    return data


def sanitize_retrieved_chunks(chunks, review_source_path="", comment_id=""):
    safe = []
    for idx, c in enumerate(chunks or []):
        if not isinstance(c, dict):
            print_flush(
                "[WARN] retrieved_chunks["
                + str(idx)
                + "] invalid type="
                + type(c).__name__
                + " review="
                + str(review_source_path)
                + " comment_id="
                + str(comment_id)
            )
            continue
        if not c.get("chunk_id"):
            print_flush(
                "[WARN] retrieved_chunks["
                + str(idx)
                + "] missing chunk_id review="
                + str(review_source_path)
                + " comment_id="
                + str(comment_id)
            )
            continue
        safe.append(c)
    return safe


def normalize_comment_audit(raw: dict, task: dict, total_paper_sections: int) -> dict:
    if not isinstance(raw, dict):
        raw = {}

    if not raw.get("support_label"):
        raw["support_label"] = raw.get("label") or raw.get("support") or raw.get("judgment") or raw.get("verdict")

    if raw.get("confidence") in (None, ""):
        raw["confidence"] = raw.get("score_confidence") or raw.get("support_confidence") or raw.get("certainty")

    if not raw.get("reasoning_summary"):
        raw["reasoning_summary"] = raw.get("explanation") or raw.get("rationale") or raw.get("justification") or raw.get("summary")

    if not raw.get("outcome"):
        raw["outcome"] = raw.get("answer") or raw.get("final_answer") or raw.get("judgment_summary")

    audit = {
        "comment_id": clean_text(raw.get("comment_id") or task.get("comment_id", "")),
        "question": clean_text(raw.get("question") or task.get("question", "")),
        "reviewer_comment": clean_text(raw.get("reviewer_comment") or task.get("reviewer_comment", "")),
        "support_label": clean_text(raw.get("support_label") or "insufficient_evidence"),
        "confidence": clamp01(raw.get("confidence", 0.0)),
        "evidence": raw.get("evidence") if isinstance(raw.get("evidence"), list) else [],
        "retrieved_chunks": [],
        "reasoning_summary": clean_text(raw.get("reasoning_summary") or ""),
        "outcome": clean_text(
            raw.get("outcome")
            or raw.get("final_answer")
            or raw.get("justification")
            or raw.get("reasoning_summary")
            or ""
        ),
        "retrieval_strength": 0.0,
        "evidence_specificity": safe_metric_float(raw.get("evidence_specificity", 0.0), 0.0),
        "section_coverage": 0.0,
    }

    clean_evidence = []
    for ev in audit["evidence"]:
        if not isinstance(ev, dict):
            continue
        clean_evidence.append(
            {
                "chunk_id": clean_text(ev.get("chunk_id") or ""),
                "section": clean_text(ev.get("section") or "unknown"),
                "page_start": safe_int(ev.get("page_start"), 0),
                "page_end": safe_int(ev.get("page_end"), 0),
                "snippet": clean_text(ev.get("snippet") or ""),
                "full_chunk_text": "",
            }
        )
    audit["evidence"] = clean_evidence[:2]

    if audit["support_label"] not in {"supported", "partially_supported", "not_supported", "insufficient_evidence"}:
        audit["support_label"] = "insufficient_evidence"

    audit["retrieved_chunks"] = []
    for c in task.get("retrieved_chunks", []) or []:
        if not isinstance(c, dict):
            continue
        if not c.get("chunk_id"):
            continue
        audit["retrieved_chunks"].append(
            {
                "chunk_id": c["chunk_id"],
                "section": c.get("section", "unknown"),
                "page_start": safe_int(c.get("page_start"), 0),
                "page_end": safe_int(c.get("page_end"), 0),
                "retrieval_score": safe_float(c.get("retrieval_score", 0.0), 0.0),
                "full_chunk_text": clean_text(c.get("full_chunk_text", "")),
            }
        )

    chunk_map = {c["chunk_id"]: c for c in audit["retrieved_chunks"] if c.get("chunk_id")}
    for ev in audit["evidence"]:
        matched = chunk_map.get(ev.get("chunk_id", ""))
        if matched:
            if not ev.get("section") or ev["section"] == "unknown":
                ev["section"] = matched.get("section", "unknown")
            if not ev.get("page_start"):
                ev["page_start"] = matched.get("page_start", 0)
            if not ev.get("page_end"):
                ev["page_end"] = matched.get("page_end", 0)
            ev["full_chunk_text"] = matched.get("full_chunk_text", "")

    audit["retrieval_strength"] = compute_retrieval_strength_from_chunks(audit["retrieved_chunks"])
    audit["section_coverage"] = compute_section_coverage_from_retrieved_chunks(
        audit["retrieved_chunks"],
        total_paper_sections=total_paper_sections,
    )

    audit["confidence"] = compute_blended_confidence(audit)
    return audit


def normalize_score_audit(raw: dict, reviewer_id: str, score_raw: str) -> dict:
    if not isinstance(raw, dict):
        raw = {}

    out = {
        "reviewer_id": clean_text(raw.get("reviewer_id") or reviewer_id),
        "given_score_raw": clean_text(raw.get("given_score_raw") or score_raw),
        "normalized_score_0_to_1": safe_float(raw.get("normalized_score_0_to_1", score_to_unit_interval(score_raw)), 0.0),
        "score_alignment": clean_text(raw.get("score_alignment") or "weakly_aligned"),
        "score_confidence": clamp01(raw.get("score_confidence", 0.0)),
        "score_justification": clean_text(raw.get("score_justification") or ""),
    }
    if out["score_alignment"] not in {"well_aligned", "mostly_aligned", "weakly_aligned", "not_aligned"}:
        out["score_alignment"] = "weakly_aligned"
    out["normalized_score_0_to_1"] = max(0.0, min(1.0, out["normalized_score_0_to_1"]))
    return out


def save_report(paper_result: dict, retrieval_rows: list[dict], top_k: int) -> dict:
    paper_id = paper_result["paper_id"]
    paper_dir = reports_dir / ("paper_" + str(paper_id))
    paper_dir.mkdir(parents=True, exist_ok=True)

    json_path = paper_dir / ("paper_" + str(paper_id) + "_audit.json")
    md_path = paper_dir / ("paper_" + str(paper_id) + "_audit.md")
    comments_jsonl = paper_dir / ("paper_" + str(paper_id) + "_comments.jsonl")
    evidence_jsonl = paper_dir / ("paper_" + str(paper_id) + "_evidence.jsonl")
    retrievals_jsonl = paper_dir / ("paper_" + str(paper_id) + "_retrievals_top" + str(top_k) + ".jsonl")
    retrievals_grouped_json = paper_dir / ("paper_" + str(paper_id) + "_retrievals_grouped_top" + str(top_k) + ".json")

    atomic_json(json_path, paper_result)

    with comments_jsonl.open("w", encoding="utf-8") as f:
        for review in paper_result["reviews"]:
            for c in review["comments"]:
                row = {
                    "paper_id": paper_id,
                    "reviewer_id": review["reviewer_id"],
                    "comment_id": c["comment_id"],
                    "question": c.get("question", ""),
                    "reviewer_comment": c["reviewer_comment"],
                    "support_label": c["support_label"],
                    "confidence": c["confidence"],
                    "retrieval_strength": c["retrieval_strength"],
                    "evidence_specificity": c["evidence_specificity"],
                    "section_coverage": c["section_coverage"],
                    "reasoning_summary": c["reasoning_summary"],
                    "outcome": c.get("outcome", ""),
                    "retrieved_chunk_count": len(c.get("retrieved_chunks", [])),
                    "evidence_count": len(c.get("evidence", [])),
                }
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with evidence_jsonl.open("w", encoding="utf-8") as f:
        for review in paper_result["reviews"]:
            for c in review["comments"]:
                for ev in c["evidence"]:
                    row = {
                        "paper_id": paper_id,
                        "reviewer_id": review["reviewer_id"],
                        "comment_id": c["comment_id"],
                        **ev,
                    }
                    f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with retrievals_jsonl.open("w", encoding="utf-8") as f:
        for row in retrieval_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    grouped_retrievals = []
    for review in paper_result["reviews"]:
        for c in review["comments"]:
            grouped_retrievals.append(
                {
                    "paper_id": paper_id,
                    "reviewer_id": review["reviewer_id"],
                    "review_source_path": review["review_source_path"],
                    "comment_id": c["comment_id"],
                    "question": c.get("question", ""),
                    "reviewer_comment": c["reviewer_comment"],
                    "retrieved_chunks": c.get("retrieved_chunks", []),
                }
            )
    atomic_json(retrievals_grouped_json, grouped_retrievals)

    lines = []
    lines.append("# Audit Report: " + str(paper_result["paper_title"]))
    lines.append("")
    lines.append("- **Paper ID:** `" + str(paper_result["paper_id"]) + "`")
    lines.append("- **Collection:** `" + str(paper_result["collection_name"]) + "`")
    lines.append("- **Top-K per comment:** " + str(top_k))
    if paper_result.get("source_file"):
        lines.append("- **Source file:** `" + str(paper_result["source_file"]) + "`")
    lines.append("")

    for review in paper_result["reviews"]:
        lines.append("---")
        lines.append("")
        lines.append("## Reviewer " + str(review["reviewer_id"]))
        lines.append("")
        lines.append("- **Review file:** `" + str(review["review_source_path"]) + "`")
        lines.append("")

        if review.get("score_audit"):
            sa = review["score_audit"]
            lines.append("### Score Audit")
            lines.append("")
            lines.append("- **Given score:** " + str(sa["given_score_raw"]))
            lines.append("- **Normalized score:** " + format(sa["normalized_score_0_to_1"], ".2f"))
            lines.append("- **Score alignment:** " + str(sa["score_alignment"]))
            lines.append("- **Score confidence:** " + format(sa["score_confidence"], ".2f"))
            lines.append("- **Score justification:** " + str(sa["score_justification"]))
            lines.append("")

        lines.append("### Comment Audits")
        lines.append("")
        for c in review["comments"]:
            lines.append("#### " + str(c["comment_id"]))
            lines.append("")
            if c.get("question"):
                lines.append("**Question:** " + str(c["question"]))
                lines.append("")
            lines.append("**Reviewer comment:**")
            lines.append("")
            lines.append("```text")
            lines.append(c["reviewer_comment"].strip())
            lines.append("```")
            lines.append("")
            lines.append("- **Support label:** " + str(c["support_label"]))
            lines.append("- **Confidence:** " + format(c["confidence"], ".2f") + " (" + confidence_band(c["confidence"]) + ")")
            lines.append("- **Retrieval strength:** " + format(c["retrieval_strength"], ".4f"))
            lines.append("- **Evidence specificity:** " + format(c["evidence_specificity"], ".4f"))
            lines.append("- **Section coverage:** " + format(c["section_coverage"], ".4f"))
            lines.append("- **Reasoning:** " + str(c["reasoning_summary"]))
            if c.get("outcome"):
                lines.append("- **Outcome:** " + str(c["outcome"]))
            lines.append("")

            if c.get("evidence"):
                lines.append("**Evidence:**")
                lines.append("")
                for ev in c["evidence"]:
                    lines.append(
                        "- `"
                        + str(ev.get("chunk_id", ""))
                        + "` | section=`"
                        + str(ev.get("section", "unknown"))
                        + "` | pages="
                        + str(ev.get("page_start", 0))
                        + "-"
                        + str(ev.get("page_end", 0))
                    )
                    if ev.get("snippet"):
                        lines.append("  - Snippet: " + str(ev["snippet"]))
                lines.append("")

    atomic_write(md_path, "\n".join(lines))

    return {
        "json_path": str(json_path.relative_to(workdir)),
        "md_path": str(md_path.relative_to(workdir)),
        "comments_jsonl": str(comments_jsonl.relative_to(workdir)),
        "evidence_jsonl": str(evidence_jsonl.relative_to(workdir)),
        "retrievals_jsonl": str(retrievals_jsonl.relative_to(workdir)),
        "retrievals_grouped_json": str(retrievals_grouped_json.relative_to(workdir)),
    }


prepared_reviews = load_prepared_reviews(workdir / "prepared_reviews.json")
TOTAL_SECTIONS_BY_PAPER = build_total_sections_by_paper(prepared_reviews)

print_flush("TOTAL_SECTIONS_BY_PAPER =", TOTAL_SECTIONS_BY_PAPER)

tokenizer, model = build_model()

results = []
errors = []
grouped = {}
grouped_retrievals = {}

print_flush("START AUDIT: total prepared reviews =", len(prepared_reviews))

for review_idx, review_payload in enumerate(prepared_reviews, start=1):
    try:
        if review_payload is None or not isinstance(review_payload, dict):
            raise ValueError("Invalid review payload at index " + str(review_idx) + ": " + type(review_payload).__name__)

        paper_id = str(review_payload["paper_id"])
        paper_title = str(review_payload["paper_title"])
        collection_name = str(review_payload["collection_name"])
        reviewer_id = str(review_payload["reviewer_id"])
        review_source_path = str(review_payload["review_source_path"])
        score_raw = str(review_payload.get("score_raw", ""))
        comment_tasks = review_payload.get("comment_tasks", [])

        print_flush(
            "\n[REVIEW "
            + str(review_idx)
            + "/"
            + str(len(prepared_reviews))
            + "] paper_id="
            + paper_id
            + " collection="
            + collection_name
            + " reviewer="
            + reviewer_id
        )

        if not comment_tasks:
            raise ValueError("No prepared comment tasks found for " + review_source_path)

        comment_audits = []
        retrieval_rows = []

        for comment_idx, task in enumerate(comment_tasks, start=1):
            if task is None or not isinstance(task, dict):
                print_flush("  [COMMENT " + str(comment_idx) + "/" + str(len(comment_tasks)) + "] invalid task -> skipped")
                continue

            task = dict(task)
            task["retrieved_chunks"] = sanitize_retrieved_chunks(
                task.get("retrieved_chunks", []),
                review_source_path=review_source_path,
                comment_id=task.get("comment_id", ""),
            )

            print_flush(
                "  [COMMENT "
                + str(comment_idx)
                + "/"
                + str(len(comment_tasks))
                + "] id="
                + str(task.get("comment_id", "__missing__"))
            )
            print_flush("    retrieved_chunks_count=" + str(len(task.get("retrieved_chunks", []) or [])))

            raw = model_generate_json(
                tokenizer=tokenizer,
                model=model,
                messages=build_comment_messages(task),
                max_input_tokens=4096,
                max_new_tokens=384,
            )
            print_flush("    RAW MODEL JSON =" + json.dumps(raw, ensure_ascii=False)[:2000])

            total_paper_sections = TOTAL_SECTIONS_BY_PAPER.get(paper_id, 1)
            audited = normalize_comment_audit(raw, task, total_paper_sections)
            comment_audits.append(audited)

            print_flush("    final_retrieval_strength=" + format(audited["retrieval_strength"], ".4f"))
            print_flush("    final_section_coverage=" + format(audited["section_coverage"], ".4f"))
            print_flush("    final_confidence=" + format(audited["confidence"], ".4f"))

            for rank, hit in enumerate(task.get("retrieved_chunks", []) or [], start=1):
                if not isinstance(hit, dict):
                    continue
                if not hit.get("chunk_id"):
                    continue

                retrieval_rows.append(
                    {
                        "paper_id": paper_id,
                        "reviewer_id": reviewer_id,
                        "review_source_path": review_source_path,
                        "comment_id": task["comment_id"],
                        "question": task.get("question", ""),
                        "reviewer_comment": task["reviewer_comment"],
                        "retrieval_query": task.get("retrieval_query", ""),
                        "retrieval_query_source": task.get("retrieval_query_source", "comment"),
                        "rank": rank,
                        "chunk_id": hit["chunk_id"],
                        "section": hit.get("section", "unknown"),
                        "page_start": safe_int(hit.get("page_start"), 0),
                        "page_end": safe_int(hit.get("page_end"), 0),
                        "retrieval_score": safe_float(hit.get("retrieval_score", 0.0), 0.0),
                        "text": clean_text(hit.get("full_chunk_text", "")),
                    }
                )

            print_flush(
                "    done -> support="
                + audited["support_label"]
                + " confidence="
                + format(audited["confidence"], ".2f")
            )

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if not comment_audits:
            raise ValueError("No valid comment audits were produced for " + review_source_path)

        score_audit = None
        if score_raw:
            print_flush("  [SCORE AUDIT] reviewer=" + reviewer_id + " score=" + score_raw)
            score_raw_obj = model_generate_json(
                tokenizer=tokenizer,
                model=model,
                messages=build_score_messages(review_payload, comment_audits),
                max_input_tokens=4096,
                max_new_tokens=384,
            )
            score_audit = normalize_score_audit(score_raw_obj, reviewer_id, score_raw)
            print_flush(
                "  [SCORE AUDIT DONE] alignment="
                + score_audit["score_alignment"]
                + " confidence="
                + format(score_audit["score_confidence"], ".2f")
            )

        bundle = {
            "paper_id": paper_id,
            "paper_title": paper_title,
            "reviewer_id": reviewer_id,
            "review_source_path": review_source_path,
            "comments": comment_audits,
            "score_audit": score_audit,
        }

        if paper_id not in grouped:
            grouped[paper_id] = {
                "paper_id": paper_id,
                "paper_title": paper_title,
                "collection_name": collection_name,
                "source_file": review_payload.get("source_file", ""),
                "reviews": [],
            }
            grouped_retrievals[paper_id] = []

        grouped[paper_id]["reviews"].append(bundle)
        grouped_retrievals[paper_id].extend(retrieval_rows)
        print_flush("AUDITED " + review_source_path)

    except Exception as e:
        tb = traceback.format_exc()
        errors.append({
            "review_file": review_payload.get("review_source_path", "__unknown__") if isinstance(review_payload, dict) else "__unknown__",
            "error": str(e),
            "traceback": tb,
        })
        print_flush("FAILED " + str(review_payload.get("review_source_path", "__unknown__") if isinstance(review_payload, dict) else "__unknown__"))
        print_flush("ERROR:", str(e))
        print_flush(tb)

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


top_k = int(prepared_reviews[0].get("top_k", 0)) if prepared_reviews and isinstance(prepared_reviews[0], dict) else 0

for paper_id, paper_result in grouped.items():
    paths = save_report(paper_result, grouped_retrievals.get(paper_id, []), top_k=top_k)
    results.append(
        {
            "paper_id": paper_id,
            "collection_name": paper_result["collection_name"],
            "review_count": len(paper_result["reviews"]),
            **paths,
        }
    )

summary = {
    "model_id": MODEL_ID,
    "processed_count": len(results),
    "error_count": len(errors),
    "results": results,
    "errors": errors,
}

detailed_answer = {
    "model_id": MODEL_ID,
    "processed_count": len(results),
    "error_count": len(errors),
    "papers": list(grouped.values()),
    "errors": errors,
}

atomic_json(summary_file, summary)
atomic_json(answer_file, detailed_answer)

if errors:
    atomic_write(status_file, "PARTIAL_OK\n")
else:
    atomic_write(status_file, "OK\n")

print_flush("DONE")
PYAUDIT

echo "[RUN] starting Qwen Transformers audit"
echo "[RUN] visible devices: HIP=$HIP_VISIBLE_DEVICES ROCR=$ROCR_VISIBLE_DEVICES CUDA=$CUDA_VISIBLE_DEVICES"

python - <<'PYDEVICECHECK'
import torch
print("[DEVICE CHECK] torch.cuda.device_count() =", torch.cuda.device_count(), flush=True)
for i in range(torch.cuda.device_count()):
    try:
        print("[DEVICE CHECK]", i, torch.cuda.get_device_name(i), flush=True)
    except Exception as e:
        print("[DEVICE CHECK]", i, e, flush=True)
PYDEVICECHECK

python "$WORKDIR/audit_runner.py"
'''

    remote_script_path = g5k.write_remote_script(remote_workdir, "run.sh", run_sh)

    jobid, submit_out = g5k.submit_job(
        remote_dir=remote_workdir,
        script_path=remote_script_path,
        gpu_request=GPU_REQUEST,
        walltime=WALLTIME,
        host_predicate=HOST_PREDICATE,
    )

    print("\n" + submit_out.strip())
    print("jobid:", jobid)

    remote_answer_path = remote_workdir + "/answer.txt"
    remote_status_path = remote_workdir + "/status.txt"

    state, answer = g5k.wait_for_answer(
        jobid=jobid,
        remote_dir=remote_workdir,
        answer_path=remote_answer_path,
        status_path=remote_status_path,
        poll_seconds=POLL_SECONDS,
        max_wait_minutes=MAX_WAIT_MINUTES,
    )

    print("\nFinal state:", state)
    print("\nRemote summary:\n", answer)

    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    local_job_dir = LOCAL_RESULTS_ROOT / ("job_" + jobid + "_" + ts)

    copied_ok = False
    try:
        g5k.copy_remote_dir_to_local(remote_workdir, local_job_dir)
        copied_ok = True
        print("\nDownloaded job folder:", local_job_dir.resolve())

        outputs_root = local_job_dir / "outputs" / "audit_reports"
        if not outputs_root.exists():
            raise RuntimeError("Missing downloaded outputs directory: " + str(outputs_root))

        print("\nAudit outputs:")
        print(outputs_root.resolve())
        for p in sorted(outputs_root.rglob("*")):
            if p.is_file():
                print(" -", p.resolve())
    finally:
        if copied_ok:
            print("\n[REMOTE] cleanup before report:", flush=True)
            print(g5k.space_report(remote_workdir).strip(), flush=True)
            g5k.cleanup_remote_job_artifacts(remote_workdir, jobid)
            print("\n[REMOTE] HOME after cleanup:", flush=True)
            print(g5k.space_report(remote_base).strip(), flush=True)
        else:
            print("\n[REMOTE] copy failed, remote workdir kept for debugging:", remote_workdir, flush=True)

    print("\nDone.")


if __name__ == "__main__":
    main()